# 🛡️ Identity Anomaly Detection — Microsoft Fabric edition

Batch scoring engine for the UEBA-style Sentinel workbook. **No labeled data needed** — unsupervised ML + statistical baselines + rules, fully explainable and SOC-consumable. Runs end-to-end on synthetic data; every cell carries its real Sentinel/KQL hook.

## Capabilities

**Core engine (§1–6)**
- **Lake-first KQL pre-aggregation** — heavy lifting in KQL (`SigninLogs`/`SecurityEvent`), one compact row per user; Python only scores aggregates.
- **Statistical baselines** — Z-score + robust-Z (median/MAD) per feature; positive spikes.
- **Isolation Forest (+SHAP)** — primary multi-dimensional rarity model (top ~2%), scale-invariant, every flag justified.
- **Composite Risk Score 0–100** — tunable weights, 5 severity bands, mirrors the workbook.
- **Fleet dashboards** — severity donut, top-N, activity heatmap.
- **Identity Protection + UEBA (§5b)** — risky users, top-by-anomaly-count, top-10 anomaly-type breakdown.
- **Export (§6)** — scored summary → Delta table `IdentityAnomalyScores` in the attached Lakehouse (readable by Real-Time Dashboards, Power BI and — via the Sentinel federated connector — by Sentinel).

**Forensic deep-dive (§7)**
- **Hunt any UPN** with an **analyst-chosen window** (`LOOKBACK_DAYS` / `DETECT_DAYS`) — risk gauge, sign-in timeline, geo/app breakdown, daily volume vs failures, first-seen novelty, raw evidence.
- **Personal baseline & deviation (§7b)** — scores an identity against **its own** history (robust-z of recent days vs personal baseline) — the core insider/compromise signal.

**Advanced detection (§8)**
- **Peer-group baselines** (behavioral K-Means), **two-tailed** detection (spikes *and* suppression), **percentile-calibrated** `PriorityScore` + confidence, **SOC narratives**, **exposure fusion** (risky identity × critical asset), **analyst feedback loop**.

**Geolocation (§9)**
- World **choropleths** (total & failed sign-ins), point-**density heatmap**, and an **hour × country** matrix exposing off-hours logins from unusual geographies.

**Next-gen hardening (§10)**
- **Concept-drift** detection — PSI metric + weekly-retrain cadence.
- **Cold-start fallback** — thin-history accounts inherit peer/global baselines.
- **Analyst-readable visuals** — 2D UMAP/t-SNE embedding + SHAP beeswarm.
- **Identity–asset graph** — node-link blast-radius for the forensic target.
- **Resource-vector peer clustering** — function-based cohorts (what they access, not how much).
- **Workload / service-account (NHE)** baselines — deterministic low-tolerance thresholds.
- **Active learning** — analyst dispositions retune IsoForest contamination/exclusion.
- **Sequential ML** — Markov event-transition modelling of action chains.

## Pipeline
| § | Step | Output |
|---|---|---|
| 1 | Extract pre-aggregated features | feature matrix |
| 2 | Z / robust-Z baselines | per-user deviations |
| 3 | Isolation Forest (+SHAP) | anomaly score + drivers |
| 4 | Composite Risk Score 0–100 | severity |
| 5 | Dashboards (donut/top-N/heatmap) | fleet overview |
| 5b | Identity Protection + UEBA anomalies | corroboration |
| 6 | Export Delta table `IdentityAnomalyScores` | dashboards + Sentinel federation |
| 7 | Per-identity forensic deep-dive (analyst-chosen window) | investigation |
| 7b | Personal baseline & deviation (user vs own history) | self-relative anomalies |
| 8 | Advanced: ensemble, peer-group, exposure fusion | `PriorityScore` + narratives |
| 9 | Geolocation analysis | choropleths + heatmaps |
| 10 | Next-gen hardening: drift/PSI, cold-start, UMAP+SHAP, identity-asset graph, resource-vector peers, NHE baselines, active learning, sequential ML | resilient UEBA platform |

> **Fabric edition.** Reads the Sentinel tables mirrored into Fabric (KQL database / Eventhouse endpoint) with the Spark Kusto connector — no MSTICPy, no Log Analytics workspace access needed. Set `USE_SENTINEL=False` to run on synthetic demo data.
>
> **Before running:** (1) attach a Lakehouse to this notebook (left pane → *Lakehouses* → *Add*) — results are written there; (2) set `KUSTO_URI` / `KUSTO_DB` in the configuration cell (Eventhouse → *Query URI* and database name/ID); (3) optional: install `shap umap-learn` (cell below, or a Fabric *Environment* for scheduled runs).
>
> **Fabric data notes:** the data-lake copy of `SigninLogs` has `Location` (country code) instead of `LocationDetails`, and no `Status` column — queries below are already adapted. Tables are OneLake shortcuts, so they are referenced as `external_table('Name')`.

In [ ]:
# Optional extras (SHAP explanations, UMAP embedding). Fabric runtime already includes pandas, numpy, scikit-learn and plotly.
# For scheduled / pipeline runs, add these packages to a Fabric Environment instead of using %pip.
%pip install -q shap umap-learn

In [ ]:
# ── Configuration (Fabric) ─────────────────────────────────────────────
import numpy as np, pandas as pd
import plotly.express as px, plotly.graph_objects as go
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
pd.set_option('display.max_columns', 50)

USE_SENTINEL = True    # True -> query the Sentinel mirror in Fabric; False -> synthetic demo data
# Eventhouse > KQL database > "Query URI" (copy from the database details pane)
KUSTO_URI    = 'https://trd-20symsby01nkxd9vmq.z1.kusto.fabric.microsoft.com'
# Database name or ID of the Sentinel mirror (e.g. Mirror_Sentinel_AllTables)
KUSTO_DB     = 'd8dabbe4-7ff9-4818-ac54-69cfbd55a354'
OUTPUT_TABLE = 'IdentityAnomalyScores'          # Delta table written to the attached Lakehouse
LOOKBACK     = 30                               # days of sign-in history used for fleet scoring

try:
    _nbu = notebookutils                      # Fabric runtime
except NameError:
    try:
        from notebookutils import mssparkutils as _nbu
    except ImportError:
        _nbu = None                           # outside Fabric (demo mode only)

def run_kql(q: str) -> pd.DataFrame:
    """Run a KQL query against the Fabric KQL database and return a pandas DataFrame."""
    token = _nbu.credentials.getToken(KUSTO_URI)
    sdf = (spark.read.format('com.microsoft.kusto.spark.synapse.datasource')
           .option('accessToken', token)
           .option('kustoCluster', KUSTO_URI)
           .option('kustoDatabase', KUSTO_DB)
           .option('kustoQuery', q)
           .load())
    return sdf.toPandas()

if USE_SENTINEL:
    print(run_kql("external_table('SigninLogs') | where TimeGenerated > ago(1d) | count"))
print('libs ready | USE_SENTINEL =', USE_SENTINEL)

## 1. Extract — pre-aggregated feature matrix
KQL does the heavy lifting and returns **one row per user per day** (sign-ins, failures, distinct IPs/countries/apps, night-time logons). We never load raw events into Python — only compact aggregates. Each column is a *behavioral feature*; the table `agg` (one row per user) is what every later step scores.

**Read it as:** higher `Failed`, `DistinctCountries`, or `Night` than a person's norm = more suspicious. Demo mode injects 3 obvious outliers so the charts have something to flag.

In [ ]:
WIN_AGG_KQL = f'''
let start={LOOKBACK}d; let end=0d;
external_table('SigninLogs')
| where TimeGenerated between (ago(start)..ago(end))
| extend Country=tostring(Location), hr=hourofday(TimeGenerated)
| summarize SignIns=count(), Failed=countif(ResultType!='0'), Success=countif(ResultType=='0'),
    DistinctIPs=dcount(IPAddress), DistinctCountries=dcount(Country), DistinctApps=dcount(AppDisplayName),
    Night=countif(hr<6 or hr>20) by UserPrincipalName, Date=format_datetime(TimeGenerated,'yyyy-MM-dd')
| extend FailRatio=todouble(Failed)/(SignIns+1)'''

def load_sentinel():
    return run_kql(WIN_AGG_KQL)

def demo_data(n_users=400, days=30, seed=42):
    rng = np.random.default_rng(seed); rows=[]
    for u in range(n_users):
        base = rng.integers(5,40)
        for d in range(days):
            s=max(0,int(rng.normal(base,base*0.3))); f=int(rng.poisson(1.5))
            rows.append([f'user{u:03d}@contoso.com', s, f, max(0,s-f), rng.integers(1,3),1,rng.integers(1,4),int(rng.poisson(0.5))])
    df=pd.DataFrame(rows,columns=['UserPrincipalName','SignIns','Failed','Success','DistinctIPs','DistinctCountries','DistinctApps','Night'])
    for u in ['user007@contoso.com','user099@contoso.com','user250@contoso.com']:
        m=df.UserPrincipalName==u; df.loc[m,['Failed','DistinctIPs','DistinctCountries','Night']]=[120,9,5,15]; df.loc[m,'SignIns']=200
    df['FailRatio']=df.Failed/(df.SignIns+1); return df

raw = load_sentinel() if USE_SENTINEL else demo_data()
if 'Date' not in raw.columns:                      # demo data has no Date column
    raw['Date'] = 'demo'
num = ['SignIns','Failed','Success','DistinctIPs','DistinctCountries','DistinctApps','Night','FailRatio']
raw[num] = raw[num].apply(pd.to_numeric, errors='coerce').fillna(0)
agg = raw.groupby('UserPrincipalName').agg(SignIns=('SignIns','sum'),Failed=('Failed','sum'),Success=('Success','sum'),DistinctIPs=('DistinctIPs','max'),DistinctCountries=('DistinctCountries','max'),DistinctApps=('DistinctApps','max'),Night=('Night','sum'),FailRatio=('FailRatio','mean')).reset_index()
agg = agg.merge(raw.groupby('UserPrincipalName').Date.nunique().rename('ActiveDays').reset_index(), on='UserPrincipalName', how='left')
print(agg.shape); agg.head()

## 2. Statistical baselines — Z and robust-Z (median + MAD)
Turns raw counts into **"how unusual"** numbers. **Z-score** = standard deviations above the population mean. **Robust-Z** uses median + MAD, so a few heavy users don't distort it. We only keep positive deviation (spikes), since only *increases* matter for threat hunting.

**Read it as:** a `_z` near 0 = normal; **>3** = strong outlier on that feature. These standardized columns feed both the risk score and the Isolation Forest.

In [ ]:
feat_cols=['SignIns','Failed','DistinctIPs','DistinctCountries','Night','FailRatio']
for c in feat_cols:
    mu,sd=agg[c].mean(),agg[c].std() or 1; agg[c+'_z']=((agg[c]-mu)/sd).clip(lower=0)
    med=agg[c].median(); mad=(agg[c]-med).abs().median() or 1; agg[c+'_rz']=(0.6745*(agg[c]-med)/mad).clip(lower=0)
z_cols=[c+'_z' for c in feat_cols]
agg[['UserPrincipalName']+z_cols].head()

## 3. Isolation Forest — multi-dimensional rarity
The primary unsupervised model. It isolates each user across all features at once; users that are easy to separate (short tree paths) are rare = anomalous. `contamination=0.02` ≈ flags the top ~2%. `iso_anomaly=-1` is an outlier; `iso_score` higher = rarer.

**Read it as:** this catches odd *combinations* (e.g. moderate failures + new country + night) that single Z-scores miss. PCA/SHAP below explain why each user was flagged.

In [ ]:
X=agg[z_cols].fillna(0).values
clf=IsolationForest(n_estimators=200,contamination=0.02,max_features=min(6,len(z_cols)),random_state=42,n_jobs=-1)
clf.fit(X); agg['iso_anomaly']=clf.predict(X); agg['iso_score']=(-clf.decision_function(X))
agg['iso_norm']=((agg.iso_score-agg.iso_score.min())/(agg.iso_score.max()-agg.iso_score.min()+1e-9))
print('outliers:',(agg.iso_anomaly==-1).sum())

## 4. Composite Risk Score 0–100
Blends ML + rules into one analyst-facing number. Weights mirror the workbook: IsoForest 30 / NewCountry 20 / Z 15 / Failures 15 / Time 10 / NewIP 10. Bands: 0–20 Normal, 21–40 Low, 41–60 Medium, 61–80 High, 81–100 Critical.

**Read it as:** sort by Score, work top-down. Score is explainable — you can always point to which signals contributed.

In [ ]:
W={'iso':30,'country':20,'z':15,'fail':15,'time':10,'ip':10}
agg['Score']=( W['iso']*agg.iso_norm + W['country']*(agg.DistinctCountries>2) + W['z']*(agg.SignIns_z>3) + W['fail']*(agg.Failed>=20) + W['time']*(agg.Night>0) + W['ip']*(agg.DistinctIPs>3) ).clip(0,100).round().astype(int)
agg['Severity']=pd.cut(agg.Score,[-1,20,40,60,80,100],labels=['Normal','Low','Medium','High','Critical'])
agg.Severity.value_counts()

In [ ]:
# SHAP feature attribution (global + local) — same explainability as the Azure SOC notebook
try:
    import shap
    expl = shap.TreeExplainer(clf); sv = expl.shap_values(agg[z_cols].fillna(0))
    imp = pd.Series(np.abs(sv).mean(0), index=z_cols).sort_values()
    title = 'SHAP global feature importance'
except Exception:
    imp = pd.Series(np.abs(StandardScaler().fit_transform(X)).mean(0), index=z_cols).sort_values()
    title = 'Feature importance (proxy — pip install shap for exact SHAP)'
px.bar(x=imp.values, y=imp.index, orientation='h', title=title).show()

## 5. Visuals
Fleet-wide overview. **Severity pie** = how many identities fall in each risk band. **Top-20 bar** = the worst identities to triage first. **Heatmap** = when activity happens (off-hours blocks stand out).

In [ ]:
cmap={'Normal':'#2ecc71','Low':'#3498db','Medium':'#f1c40f','High':'#e67e22','Critical':'#e74c3c'}
fig=px.pie(agg,names='Severity',hole=.55,color='Severity',color_discrete_map=cmap,title='Severity distribution')
fig.update_traces(textinfo='label+value'); fig.show()

In [ ]:
top=agg.sort_values('Score',ascending=False).head(20)
px.bar(top,x='Score',y='UserPrincipalName',orientation='h',color='Severity',color_discrete_map=cmap,title='Top 20 anomalous identities').update_layout(yaxis=dict(autorange='reversed')).show()

In [ ]:
# Activity heatmap (hour x weekday) — real data when USE_SENTINEL, synthetic otherwise
if USE_SENTINEL:
    hm = run_kql(f'''external_table('SigninLogs') | where TimeGenerated > ago({LOOKBACK}d)
| summarize SignIns=count() by Day=dayofweek(TimeGenerated)/1d, Hour=hourofday(TimeGenerated)''')
    names = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']
    h = (hm.assign(Day=hm.Day.astype(int).map(dict(enumerate(names))))
           .pivot_table(index='Day', columns='Hour', values='SignIns', fill_value=0)
           .reindex(['Mon','Tue','Wed','Thu','Fri','Sat','Sun']).reindex(columns=range(24), fill_value=0).fillna(0))
else:
    h=pd.DataFrame(np.random.default_rng(1).poisson(3,(7,24)),index=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])
px.imshow(h,labels=dict(x='Hour (UTC)',y='Day',color='Sign-ins'),title='Activity heatmap (hour × day)',color_continuous_scale='RdYlGn_r').show()

## 5b. Identity Protection risk & top Sentinel anomalies
External corroboration for our scores. **Risky users** = Entra ID Identity Protection verdicts (low/medium/high). **Top by anomaly count** = who has the most native Sentinel UEBA `Anomalies`. **Stacked top-10** = *which* anomaly templates each user triggered (brute force, mass download, new geo, etc.).

**Read it as:** a user flagged here **and** scored high by us = high-confidence priority.

In [ ]:
# Identity Protection — risky users (AADUserRiskEvents). Demo fallback if not querying Sentinel.
RISK_KQL = '''external_table('AADUserRiskEvents') | where TimeGenerated > ago(30d)
| summarize Events=count(), MaxRisk=max(RiskLevel) by UserPrincipalName=tolower(UserPrincipalName), RiskLevel
| summarize Events=sum(Events) by UserPrincipalName, RiskLevel | top 30 by Events desc'''

def demo_risk():
    rng = np.random.default_rng(3); u = agg.UserPrincipalName.sample(20, random_state=3).tolist()
    lv = rng.choice(['low','medium','high'], 20, p=[.5,.3,.2])
    return pd.DataFrame({'UserPrincipalName': u, 'RiskLevel': lv, 'Events': rng.integers(1, 25, 20)})

risk = run_kql(RISK_KQL) if USE_SENTINEL else demo_risk()
if risk.empty: print('No AADUserRiskEvents rows in the window — using demo data for the chart'); risk = demo_risk()
order = {'high': 3, 'medium': 2, 'low': 1}
risk = risk.sort_values('Events', ascending=False).head(20)
px.bar(risk, x='Events', y='UserPrincipalName', orientation='h', color='RiskLevel',
       color_discrete_map={'high':'#e74c3c','medium':'#e67e22','low':'#f1c40f'},
       title='Entra ID Identity Protection — risky users (30d)').update_layout(yaxis=dict(autorange='reversed')).show()

In [ ]:
# Which identities have the most Sentinel UEBA anomalies
ANOM_KQL = '''external_table('Anomalies') | where TimeGenerated > ago(30d)
| extend UPN=tolower(coalesce(UserPrincipalName, UserName))
| where isnotempty(UPN)
| summarize Anomalies=count(), AvgScore=round(avg(todouble(Score)),2), Techniques=make_set(tostring(Techniques),5) by UPN
| top 20 by Anomalies desc'''

def demo_anoms():
    rng = np.random.default_rng(5); u = agg.sort_values('Score', ascending=False).UserPrincipalName.head(15).tolist()
    return pd.DataFrame({'UPN': u, 'Anomalies': sorted(rng.integers(1,18,15), reverse=True), 'AvgScore': rng.uniform(.4,.95,15).round(2)})

an = run_kql(ANOM_KQL) if USE_SENTINEL else demo_anoms()
if an.empty: print('No Anomalies rows (UEBA anomalies not enabled?) — using demo data for the chart'); an = demo_anoms()
px.bar(an.head(20), x='Anomalies', y='UPN', orientation='h', color='AvgScore', color_continuous_scale='Reds',
       title='Top identities by Sentinel UEBA anomaly count (30d)').update_layout(yaxis=dict(autorange='reversed')).show()
an.head()

In [ ]:
# Top 10 users: WHICH UEBA anomaly templates each one has
TOP10_KQL = '''let top=external_table('Anomalies') | where TimeGenerated > ago(30d)
| extend UPN=tolower(coalesce(UserPrincipalName,UserName)) | where isnotempty(UPN)
| summarize c=count() by UPN | top 10 by c desc | project UPN;
external_table('Anomalies') | where TimeGenerated > ago(30d)
| extend UPN=tolower(coalesce(UserPrincipalName,UserName)) | where UPN in (top)
| summarize Count=count(), MaxScore=round(max(todouble(Score)),2) by UPN, AnomalyTemplate=AnomalyTemplateName
| order by Count desc'''

def demo_top10():
    rng = np.random.default_rng(8)
    tmpl = ['Anomalous sign-in','Mass download','New geo location','Unusual logon time','Brute force','New device','Privilege escalation']
    u = agg.sort_values('Score', ascending=False).UserPrincipalName.head(10).tolist()
    rows = [[x, t, int(rng.integers(1,8)), round(rng.uniform(.4,.95),2)] for x in u for t in rng.choice(tmpl, rng.integers(2,4), replace=False)]
    return pd.DataFrame(rows, columns=['UPN','AnomalyTemplate','Count','MaxScore'])

t10 = run_kql(TOP10_KQL) if USE_SENTINEL else demo_top10()
if t10.empty: print('No Anomalies rows — using demo data for the chart'); t10 = demo_top10()
order = t10.groupby('UPN').Count.sum().sort_values(ascending=False).index.tolist()
px.bar(t10, x='UPN', y='Count', color='AnomalyTemplate', category_orders={'UPN': order},
       title='Top 10 users — UEBA anomaly types (stacked)').update_layout(xaxis_tickangle=-30).show()
t10.pivot_table(index='UPN', columns='AnomalyTemplate', values='Count', aggfunc='sum', fill_value=0)

## 6. Export → Delta table `IdentityAnomalyScores` (Lakehouse)
Writes the compact scored summary (one row per identity per run) to the **attached Lakehouse** as a Delta table. From there:
- **Real-Time Dashboards / Power BI** read it directly (or through an Eventhouse OneLake shortcut).
- **Microsoft Sentinel** reads it through the **federated connector** (Defender portal → Data connectors → Data federation → Microsoft Fabric) as `IdentityAnomalyScores_<instance>` — so analytics rules and hunting can join on the scores. The table has a `TimeGenerated` column so the Sentinel time picker works.
- Each run is appended with a `RunId`, which §10b uses for real drift (PSI) monitoring.

In [ ]:
import datetime as _dt
RUN_ID = _dt.datetime.utcnow().strftime('%Y%m%d%H%M%S')
out=agg[['UserPrincipalName','Score','Severity','SignIns','Failed','DistinctIPs','DistinctCountries','iso_score']].copy()
out['Severity']=out['Severity'].astype(str)
out['TimeGenerated']=pd.Timestamp.utcnow().tz_localize(None)
out['RunId']=RUN_ID
for c in ['Score','SignIns','Failed','DistinctIPs','DistinctCountries']: out[c]=out[c].astype('int64')
out['iso_score']=out['iso_score'].astype(float)

if USE_SENTINEL:
    (spark.createDataFrame(out).write.format('delta').mode('append')
          .option('mergeSchema', 'true').saveAsTable(OUTPUT_TABLE))
    print(f'appended {len(out)} rows -> Lakehouse table {OUTPUT_TABLE} (RunId {RUN_ID})')
else:
    out.to_csv('IdentityAnomalyScores.csv',index=False)
    print('demo mode: exported', len(out), 'rows -> IdentityAnomalyScores.csv')
out.sort_values('Score',ascending=False).head()

## 7. 🔎 Forensic hunting — single identity
Deep-dive one user. Set `MANUAL_UPN` to investigate **any** identity (even if not in the
top scores); blank = top scorer. **You choose the time window** via `LOOKBACK_DAYS` /
`DETECT_DAYS` in the first cell. Renders: risk gauge, sign-in timeline (red=failed),
geo/app breakdown, daily volume vs failures, first-seen novelty, and a **personal
baseline & deviation** view (§7b) that scores this identity against *its own* history.

**Read it as:** the timeline answers *what changed and when*; §7b answers *is this person
acting unlike themselves over the days you chose*; the gauge gives the overall verdict.

In [ ]:
# Set MANUAL_UPN to investigate ANY identity (even if not scored above); else defaults to top scorer
MANUAL_UPN = ''   # e.g. 'someone@contoso.com' — blank = top scorer
TARGET_USER = MANUAL_UPN.strip() or agg.sort_values('Score', ascending=False).UserPrincipalName.iloc[0]

# ── Analyst-chosen time window (change these freely) ────────────────────
LOOKBACK_DAYS = 30      # total days of history to pull for this identity
DETECT_DAYS   = 3       # most-recent days treated as the "detection" window
BASELINE_DAYS = max(1, LOOKBACK_DAYS - DETECT_DAYS)   # earlier days = this user's OWN baseline
print(f'Forensic target: {TARGET_USER}  |  lookback {LOOKBACK_DAYS}d  (baseline {BASELINE_DAYS}d + detect {DETECT_DAYS}d)')

RAW_KQL = f"""union external_table('SigninLogs'), external_table('AADNonInteractiveUserSignInLogs')
| where TimeGenerated > ago({LOOKBACK_DAYS}d) | where UserPrincipalName =~ '{TARGET_USER}'
| extend G = geo_info_from_ip_address(IPAddress)
| project TimeGenerated, IPAddress, Country=iff(isempty(tostring(Location)), tostring(G.country), tostring(Location)), City=tostring(G.city),
          Lat=todouble(G.latitude), Lon=todouble(G.longitude), AppDisplayName, ResultType=tostring(ResultType), Status=tostring(ResultDescription)"""

def synth_user_events(seed=7, days=LOOKBACK_DAYS, per_day=10):
    rng = np.random.default_rng(seed); n = int(days * per_day)
    t = pd.to_datetime(pd.Timestamp.utcnow()) - pd.to_timedelta(rng.uniform(0, days, n), unit='D')
    df = pd.DataFrame({'TimeGenerated': t,
        'IPAddress': rng.choice(['10.0.0.5','10.0.0.7','185.23.4.9','45.66.7.8'], n, p=[.6,.2,.12,.08]),
        'Country': rng.choice(['US','US','US','RU','NG'], n), 'City': '-',
        'AppDisplayName': rng.choice(['Office365','Teams','Azure Portal','Legacy SMTP'], n),
        'ResultType': rng.choice(['0','0','0','50126'], n), 'Status': '-'}).sort_values('TimeGenerated')
    # demo: inject a behavioral spike in the detection window so deviations are visible
    recent = df.TimeGenerated >= (df.TimeGenerated.max() - pd.Timedelta(days=DETECT_DAYS))
    spike = df[recent].sample(frac=0.5, random_state=seed).index
    df.loc[spike, 'ResultType'] = '50126'
    df.loc[spike, 'Country']    = rng.choice(['RU','NG','IR'], len(spike))
    df.loc[spike, 'IPAddress']  = rng.choice(['91.219.1.7','103.4.5.6'], len(spike))
    return df

ev = run_kql(RAW_KQL) if USE_SENTINEL else synth_user_events()
if ev.empty: raise ValueError(f'No sign-ins for {TARGET_USER} in the last {LOOKBACK_DAYS} days')
ev['TimeGenerated'] = pd.to_datetime(ev['TimeGenerated']).dt.tz_localize(None); ev['Failed'] = ev.ResultType.astype(str) != '0'
print(len(ev), 'events'); ev.head()

In [ ]:
m = agg[agg.UserPrincipalName == TARGET_USER]
if len(m):
    r = m.iloc[0]; score = int(r.Score)
    print(f"RISK {score}/100  ·  {r.Severity}  ·  SignIns {r.SignIns}  ·  Failed {r.Failed}  ·  IPs {r.DistinctIPs}  ·  Countries {r.DistinctCountries}")
else:
    score = 0; print(f"{TARGET_USER} not in scored set — ad-hoc forensic only (no composite score).")
gauge = go.Figure(go.Indicator(mode='gauge+number', value=score, title={'text': f'Risk · {TARGET_USER}'},
    gauge={'axis': {'range': [0, 100]}, 'bar': {'color': '#e74c3c'},
           'steps': [{'range': [0,20],'color':'#2ecc71'},{'range':[20,40],'color':'#3498db'},{'range':[40,60],'color':'#f1c40f'},{'range':[60,80],'color':'#e67e22'},{'range':[80,100],'color':'#e74c3c'}]}))
gauge.show()

In [ ]:
# Sign-in timeline: IP vs time, colored by outcome, shaped by country
px.scatter(ev, x='TimeGenerated', y='IPAddress', color=ev.Failed.map({True:'Failed', False:'Success'}),
           symbol='Country', color_discrete_map={'Failed':'#e74c3c','Success':'#2ecc71'},
           title=f'Sign-in timeline — {TARGET_USER}').show()

In [ ]:
# Geo + app breakdown
geo = ev.Country.value_counts().reset_index(); geo.columns=['Country','Count']
app = ev.AppDisplayName.value_counts().reset_index(); app.columns=['App','Count']
px.bar(geo, x='Country', y='Count', color='Count', color_continuous_scale='RdYlGn_r', title='Sign-ins by country').show()
px.bar(app, x='App', y='Count', title='Sign-ins by app').show()

In [ ]:
# Daily volume vs failures + novelty + raw evidence
daily = ev.set_index('TimeGenerated').resample('D').agg(SignIns=('IPAddress','size'), Failed=('Failed','sum')).reset_index()
px.bar(daily, x='TimeGenerated', y=['SignIns','Failed'], barmode='group', title=f'Daily volume vs failures — {TARGET_USER}').show()
cut = ev.TimeGenerated.max() - pd.Timedelta(days=DETECT_DAYS)
hist, recent = ev[ev.TimeGenerated < cut], ev[ev.TimeGenerated >= cut]
print('New IPs:', sorted(set(recent.IPAddress) - set(hist.IPAddress)))
print('New countries:', sorted(set(recent.Country) - set(hist.Country)))
ev.sort_values('TimeGenerated', ascending=False).head(50)

### 7b. Personal baseline & deviation — this identity vs **its own** normal
The fleet score (§4) and peer score (§8a) ask *"is this user unlike others?"*. This
asks *"is this user unlike **themselves**?"* — the strongest insider/compromise signal.

It splits the analyst-chosen `LOOKBACK_DAYS` into a **baseline** (earlier `BASELINE_DAYS`)
and a **detection** window (most-recent `DETECT_DAYS`), builds the user's per-day behavior,
and robust-z-scores each recent day against that personal baseline (median + MAD).

**Read it as:** a bar past the dashed ±3σ line = that day deviates sharply from this
person's own routine. Change `LOOKBACK_DAYS` / `DETECT_DAYS` in the cell above to retune.

In [ ]:
# ── §7b: Personal baseline + deviation (user vs their OWN history) ──
ev['Date'] = ev.TimeGenerated.dt.floor('D')
per_day = ev.groupby('Date').agg(
    SignIns=('IPAddress', 'size'),
    Failed=('Failed', 'sum'),
    DistinctIPs=('IPAddress', 'nunique'),
    DistinctCountries=('Country', 'nunique'),
    Night=('TimeGenerated', lambda s: int(((s.dt.hour < 6) | (s.dt.hour > 20)).sum())),
).reset_index()

split  = ev.TimeGenerated.max().floor('D') - pd.Timedelta(days=DETECT_DAYS - 1)
base   = per_day[per_day.Date <  split]      # the user's OWN baseline period
detect = per_day[per_day.Date >= split]      # most-recent detection window
metrics = ['SignIns', 'Failed', 'DistinctIPs', 'DistinctCountries', 'Night']

if len(base) and len(detect):
    rows = []
    for mcol in metrics:
        med = base[mcol].median()
        mad = (base[mcol] - med).abs().median()
        scale = mad if mad > 0 else (base[mcol].std() or 1.0)   # avoid divide-by-zero on constant baselines
        for _, d in detect.iterrows():
            z = float(np.clip(0.6745 * (d[mcol] - med) / scale, -10, 10))   # cap to a sane range
            rows.append({'Date': d.Date.date(), 'Metric': mcol, 'Value': d[mcol],
                         'BaselineMedian': round(med, 2), 'DeviationZ': round(z, 2)})
    dev = pd.DataFrame(rows)
    flagged = dev[dev.DeviationZ.abs() >= 3]
    print(f'Baseline {len(base)}d  |  Detection {len(detect)}d  |  deviating metric-days (|z|>=3): {len(flagged)}')
    if len(flagged):
        print(flagged.to_string(index=False))

    fig = px.bar(dev, x='Metric', y='DeviationZ', color=dev.Date.astype(str), barmode='group',
                 title=f'Personal-baseline deviation (robust z) — {TARGET_USER}',
                 labels={'DeviationZ': "σ from this user's own baseline", 'color': 'Day'})
    fig.add_hline(y=3,  line=dict(color='red', dash='dash'), annotation_text='anomaly threshold (3σ)')
    fig.add_hline(y=-3, line=dict(color='red', dash='dash'))
    fig.update_layout(height=460)
    fig.show()
else:
    print('Not enough days for a baseline/detection split — raise LOOKBACK_DAYS or lower DETECT_DAYS.')

## 8. 🚀 Advanced improvements
Production-hardening on top of v1: **peer-group baselines**, **two-tailed** detection (spikes *and* suppression), **percentile calibration**, **human-readable narratives**, an **analyst feedback loop**, and **exposure fusion** (boost when a risky identity meets a vulnerable/critical asset). Each cell adds columns to `agg` — no rework of v1.

### 8a. Peer-group baselines — score each identity against its cohort
KMeans clusters identities into behavioral peer groups, then computes a robust
(median/MAD) deviation **within each group**. Catches users who look normal
fleet-wide but are outliers *for their role*. Adds `PeerGroup`, `PeerZ`.

In [ ]:
# ── §8a: Peer-group baselines ──
from sklearn.cluster import KMeans
_num = [c for c in feat_cols if c in agg.columns]
_Zp  = StandardScaler().fit_transform(agg[_num].fillna(0).values)
_k   = max(2, min(6, (len(agg) // 8) or 2))
agg['PeerGroup'] = KMeans(n_clusters=_k, n_init=10, random_state=42).fit_predict(_Zp)

# robust within-group deviation (median + MAD), positive spikes only
_pz = {}
for col in _num:
    med = agg.groupby('PeerGroup')[col].transform('median')
    mad = (agg[col] - med).abs().groupby(agg['PeerGroup']).transform('median').replace(0, 1e-9)
    _pz[col] = (0.6745 * (agg[col] - med) / mad).clip(-10, 10)
agg['PeerZ'] = pd.DataFrame(_pz).clip(lower=0).max(axis=1).round(2)

print(f'Peer groups: {_k} | mean PeerZ = {agg.PeerZ.mean():.2f} | max = {agg.PeerZ.max():.2f}')
agg[['UserPrincipalName', 'PeerGroup', 'PeerZ']].sort_values('PeerZ', ascending=False).head(8)

### 8b. Two-tailed detection — spikes **and** suppression
Insider threats often *reduce* activity (recon, data hoarding off-hours) rather
than spike it. We track absolute robust-Z (deviation in either direction) plus a
suppression flag for accounts that go unusually quiet while still failing auth.
Adds `TwoTailedZ`, `SuppressionFlag`.

In [ ]:
# ── §8b: Two-tailed detection (spikes + suppression) ──
_tt = {}
for col in _num:
    med = agg[col].median()
    mad = (agg[col] - med).abs().median() or 1e-9
    _tt[col] = (0.6745 * (agg[col] - med) / mad).abs()
agg['TwoTailedZ'] = pd.DataFrame(_tt).max(axis=1).clip(0, 10).round(2)

_signins = agg['SignIns'] if 'SignIns' in agg else pd.Series(0, index=agg.index)
_failed  = agg['Failed']  if 'Failed'  in agg else pd.Series(0, index=agg.index)
agg['SuppressionFlag'] = ((_signins < _signins.median() * 0.2) & (_failed > 0)).astype(int)

print(f'Two-tailed mean = {agg.TwoTailedZ.mean():.2f} | suppression flags = {int(agg.SuppressionFlag.sum())}')
agg[['UserPrincipalName', 'TwoTailedZ', 'SuppressionFlag']].sort_values('TwoTailedZ', ascending=False).head(8)

### 8d. Percentile calibration → unified `PriorityScore`
Raw scores aren't comparable across runs. We percentile-rank the **Isolation-Forest**
score into `CalibratedRisk` (0–100) and fuse it with v1 `Score` and peer deviation into a
single analyst-facing `PriorityScore`, with a `Confidence` tier from how many independent
signals agree (IsoForest outlier, high calibrated risk, peer spike).

In [ ]:
# ── §8d: Percentile calibration + PriorityScore ──
agg['CalibratedRisk'] = (agg['iso_norm'].rank(pct=True) * 100).round(1)
agg['PriorityScore'] = (
    0.55 * agg['Score']
    + 0.30 * agg['CalibratedRisk']
    + 0.15 * (agg['PeerZ'].clip(0, 10) * 10)
).clip(0, 100).round(1)
# confidence = how many independent signals agree (IsoForest outlier, high calibrated risk, peer spike)
_agree = ((agg['iso_anomaly'] == -1).astype(int)
          + (agg['CalibratedRisk'] >= 80).astype(int)
          + (agg['PeerZ'] >= 3).astype(int))
agg['Confidence'] = np.where(_agree >= 2, 'High', np.where(_agree == 1, 'Medium', 'Low'))
agg[['UserPrincipalName', 'Score', 'CalibratedRisk', 'PeerZ', 'PriorityScore', 'Confidence']] \
    .sort_values('PriorityScore', ascending=False).head(10)

### 8e. Human-readable narratives
Turns the numeric drivers into a one-line SOC summary per identity, so an analyst
can triage without reading the feature matrix. Adds `Narrative`.

In [ ]:
# ── §8e: Human-readable SOC narratives ──
def _narrative(r):
    bits = []
    if r.get('Failed', 0) > 0:            bits.append(f"{int(r['Failed'])} failed sign-ins")
    if r.get('DistinctCountries', 0) > 1: bits.append(f"{int(r['DistinctCountries'])} countries")
    if r.get('DistinctIPs', 0) > 0:       bits.append(f"{int(r['DistinctIPs'])} IPs")
    if r.get('Night', 0) > 0:             bits.append(f"{int(r['Night'])} off-hours events")
    if r.get('PeerZ', 0) > 3:             bits.append(f"{r['PeerZ']:.1f}σ above peer group {int(r.get('PeerGroup', 0))}")
    if r.get('SuppressionFlag', 0):       bits.append('activity suppression')
    if r.get('iso_anomaly', 0) == -1:   bits.append('Isolation-Forest outlier')
    drivers = ', '.join(bits) if bits else 'no strong drivers'
    return f"[{r.get('Confidence', '?')}] {r['UserPrincipalName']} — priority {r.get('PriorityScore', 0):.0f}/100. Drivers: {drivers}."

agg['Narrative'] = agg.apply(_narrative, axis=1)
for n in agg.sort_values('PriorityScore', ascending=False).head(6).Narrative:
    print('•', n)

### 8f. Exposure fusion — risky identity × critical asset
A medium-risk user touching a crown-jewel asset outranks a high-risk user on a
sandbox. We fuse identity risk with asset criticality (demo values — replace with
your CMDB / Defender exposure join). Adds `AssetCriticality`, `ExposureScore`,
`CrownJewelContact`.

In [ ]:
# ── §8f: Exposure fusion (identity risk × privilege / asset criticality) ──
# Real data: criticality from Entra roles in IdentityInfo (3 = tier-0 role, 2 = other admin role, 1 = any role, 0 = none).
# Demo mode keeps the original random values.
if USE_SENTINEL:
    roles = run_kql('''external_table('IdentityInfo') | where TimeGenerated > ago(14d)
| summarize arg_max(TimeGenerated, AssignedRoles) by UPN=tolower(AccountUPN)
| extend R = tostring(AssignedRoles)
| extend AssetCriticality = case(R has_any ("Global Administrator","Privileged Role Administrator","Privileged Authentication Administrator","Security Administrator","Exchange Administrator","Application Administrator","Cloud Application Administrator"), 3,
                                 R has "Administrator", 2, R != "[]" and isnotempty(R), 1, 0)
| project UPN, AssetCriticality''')
    agg['AssetCriticality'] = agg.UserPrincipalName.str.lower().map(dict(zip(roles.UPN, roles.AssetCriticality))).fillna(0).astype(int)
else:
    _rng = np.random.default_rng(7)
    agg['AssetCriticality']  = _rng.choice([0, 1, 2, 3], size=len(agg), p=[0.55, 0.25, 0.13, 0.07])
agg['ExposureScore']     = (agg['PriorityScore'] * (1 + 0.15 * agg['AssetCriticality'])).clip(0, 100).round(1)
agg['CrownJewelContact'] = (agg['AssetCriticality'] >= 3).astype(int)

_top20 = agg.sort_values('PriorityScore', ascending=False).head(20)
print(f'Tier-0 identities in top-20 priority: {int(_top20.CrownJewelContact.sum())}')
agg[['UserPrincipalName', 'PriorityScore', 'AssetCriticality', 'ExposureScore']] \
    .sort_values('ExposureScore', ascending=False).head(8)

### 8g. Analyst feedback loop
Capture triage verdicts (`tp` / `fp` / `benign`). The notebook learns a
false-positive ceiling and suppresses similar unreviewed alerts on the next run —
a lightweight, label-free way to reduce alert fatigue. Adds `AnalystVerdict`,
`AdjExposure`.

In [ ]:
# ── §8g: Analyst feedback loop ──
FEEDBACK = {            # UPN -> 'tp' | 'fp' | 'benign'
    # 'user007@contoso.com': 'tp',
}
agg['AnalystVerdict'] = agg['UserPrincipalName'].map(FEEDBACK).fillna('unreviewed')

_fp = agg[agg.AnalystVerdict == 'fp']
if len(_fp):
    _ceiling = _fp.ExposureScore.max()
    agg['AdjExposure'] = np.where(
        (agg.AnalystVerdict == 'unreviewed') & (agg.ExposureScore <= _ceiling),
        agg.ExposureScore * 0.7, agg.ExposureScore).round(1)
    print(f'Learned FP ceiling {_ceiling:.0f} — dampened similar unreviewed alerts by 30%.')
else:
    agg['AdjExposure'] = agg['ExposureScore']
    print('No analyst feedback yet — add verdicts to FEEDBACK to auto-tune thresholds.')
print(agg.AnalystVerdict.value_counts().to_dict())

### 8h. Final analyst priority queue
Everything fuses into one ranked, color-coded queue — exposure-weighted and
feedback-adjusted — the single view a SOC analyst works top-down.

In [ ]:
# ── §8h: Final prioritized queue ──
_q = agg.sort_values('AdjExposure', ascending=False).head(20)
fig = px.bar(_q, x='AdjExposure', y='UserPrincipalName', orientation='h', color='Confidence',
             color_discrete_map={'High': '#e74c3c', 'Medium': '#f39c12', 'Low': '#3498db'},
             hover_data=['Score', 'CalibratedRisk', 'PeerZ', 'AssetCriticality'],
             title='§8 — Final analyst priority queue (exposure-fused, feedback-adjusted)')
fig.update_layout(yaxis=dict(autorange='reversed'), height=620, margin=dict(l=10, r=10, t=50, b=10))
fig.show()

## 9. 🌍 Geolocation analysis + heatmaps
Geographic view of where sign-ins originate. Renders four heatmaps: a **world choropleth** of total sign-ins by country, a **failed-login choropleth** highlighting attack origins, a **density map** of point intensity (true geographic heatmap), and an **hour × country** matrix exposing off-hours logins from unusual geographies.

Reuses the forensic `ev` (single identity) if §7 has run, otherwise synthesises fleet-wide data. In Fabric, §7 already adds `Lat`/`Lon` from `geo_info_from_ip_address()` (the lake copy has no `LocationDetails`), so the density map uses real IP geolocation.

**Read it as:** unexpected countries on the choropleths, hot spots on the density map, or off-hours blocks in the hour × country grid = priority geolocation anomalies.

In [ ]:
# ── §9: Geolocation analysis + heatmaps ──────────────────────────────
# Self-contained imports so this cell runs even if the imports cell hasn't been run yet.
import numpy as np, pandas as pd
import plotly.express as px

# Approximate country centroids (ISO3, name, lat, lon). Extend as needed; unknown
# codes fall back to pycountry (if installed) for ISO3/name resolution.
COUNTRY_GEO = {
    'US': ('USA', 'United States',  39.8,  -98.6), 'RU': ('RUS', 'Russia',          61.5,  105.3),
    'NG': ('NGA', 'Nigeria',         9.1,    8.7), 'GB': ('GBR', 'United Kingdom',  54.0,   -2.0),
    'DE': ('DEU', 'Germany',        51.2,   10.4), 'FR': ('FRA', 'France',          46.2,    2.2),
    'CN': ('CHN', 'China',          35.9,  104.2), 'IN': ('IND', 'India',           22.0,   79.0),
    'BR': ('BRA', 'Brazil',        -14.2,  -51.9), 'IR': ('IRN', 'Iran',            32.4,   53.7),
    'KP': ('PRK', 'North Korea',    40.3,  127.5), 'CA': ('CAN', 'Canada',          56.1, -106.3),
    'JP': ('JPN', 'Japan',          36.2,  138.3), 'IE': ('IRL', 'Ireland',         53.4,   -8.2),
    'NL': ('NLD', 'Netherlands',    52.1,    5.3), 'UA': ('UKR', 'Ukraine',         48.4,   31.2),
    'AU': ('AUS', 'Australia',     -25.3,  133.8), 'SG': ('SGP', 'Singapore',        1.35, 103.8),
    'KR': ('KOR', 'South Korea',    35.9,  127.8), 'TR': ('TUR', 'Turkey',          38.9,   35.2),
}

def _resolve(code):
    cc = str(code).strip().upper()
    if cc in COUNTRY_GEO:
        return COUNTRY_GEO[cc]
    try:
        import pycountry
        rec = pycountry.countries.get(alpha_2=cc) or pycountry.countries.lookup(cc)
        if rec:
            return (rec.alpha_3, rec.name, None, None)
    except Exception:
        pass
    return (None, cc, None, None)

# Source events: reuse forensic `ev` if present, else synthesize fleet-wide geo events
if 'ev' in dir():
    gdf = ev.copy()
    geo_scope = TARGET_USER if 'TARGET_USER' in dir() else 'forensic target'
else:
    _rng = np.random.default_rng(11); _n = 1500
    gdf = pd.DataFrame({
        'TimeGenerated': pd.date_range(end=pd.Timestamp.utcnow(), periods=_n, freq='30min'),
        'IPAddress': [f'{a}.{b}.{c}.{d}' for a, b, c, d in _rng.integers(1, 254, (_n, 4))],
        'Country': _rng.choice(['US','US','US','GB','DE','RU','NG','IR','CN','BR'], _n),
        'ResultType': _rng.choice(['0','0','0','50126'], _n)})
    geo_scope = 'fleet (synthetic)'

if 'Failed' not in gdf.columns:
    gdf['Failed'] = gdf.get('ResultType', '0').astype(str) != '0'

# Resolve ISO3 + name + centroid lat/lon
_res = gdf.Country.apply(_resolve)
gdf['ISO3']        = _res.apply(lambda t: t[0])
gdf['CountryName'] = _res.apply(lambda t: t[1])
if {'Lat', 'Lon'}.issubset(gdf.columns):                      # real Sentinel geoCoordinates
    gdf['Lat'] = pd.to_numeric(gdf['Lat'], errors='coerce')
    gdf['Lon'] = pd.to_numeric(gdf['Lon'], errors='coerce')
else:                                                          # centroid + jitter for spread
    _j = np.random.default_rng(3)
    gdf['Lat'] = pd.to_numeric(_res.apply(lambda t: t[2]), errors='coerce') + _j.normal(0, 1.5, len(gdf))
    gdf['Lon'] = pd.to_numeric(_res.apply(lambda t: t[3]), errors='coerce') + _j.normal(0, 1.5, len(gdf))

# Per-country aggregation
by_country = (gdf.groupby(['ISO3', 'CountryName'])
                 .agg(SignIns=('Country', 'size'), Failed=('Failed', 'sum'))
                 .reset_index())
by_country['FailRate'] = (by_country.Failed / by_country.SignIns * 100).round(1)

# 1) Choropleth heatmap - total sign-ins by country
px.choropleth(by_country.dropna(subset=['ISO3']), locations='ISO3', color='SignIns',
              hover_name='CountryName',
              hover_data={'SignIns': True, 'Failed': True, 'FailRate': True, 'ISO3': False},
              color_continuous_scale='YlOrRd', projection='natural earth',
              title=f'🌍 Sign-in heatmap by country - {geo_scope}').show()

# 2) Choropleth heatmap - failed sign-ins (attack origins)
_fc = by_country[(by_country.Failed > 0)].dropna(subset=['ISO3'])
if len(_fc):
    px.choropleth(_fc, locations='ISO3', color='Failed', hover_name='CountryName',
                  hover_data={'Failed': True, 'FailRate': True, 'ISO3': False},
                  color_continuous_scale='Reds', projection='natural earth',
                  title=f'🚨 Failed sign-in heatmap (attack origins) - {geo_scope}').show()

# 3) Density heatmap on map (point intensity)
_gm = gdf.dropna(subset=['Lat', 'Lon'])
if len(_gm):
    try:
        px.density_mapbox(_gm, lat='Lat', lon='Lon', radius=12,
                          center=dict(lat=20, lon=0), zoom=0.6, mapbox_style='carto-positron',
                          color_continuous_scale='Turbo',
                          title=f'🔥 Sign-in density heatmap - {geo_scope}').show()
    except Exception:                                          # plotly version fallback
        px.scatter_geo(_gm, lat='Lat', lon='Lon', color='Failed', opacity=0.5,
                       projection='natural earth',
                       color_discrete_map={True: '#e74c3c', False: '#3498db'},
                       title=f'🔥 Sign-in locations - {geo_scope}').show()

# 4) Hour x country temporal heatmap (off-hours logins from unusual geos)
gdf['Hour'] = pd.to_datetime(gdf.TimeGenerated).dt.hour
_pivot = gdf.pivot_table(index='CountryName', columns='Hour', values='IPAddress',
                         aggfunc='size', fill_value=0)
px.imshow(_pivot, color_continuous_scale='Inferno', aspect='auto',
          labels=dict(x='Hour of day (UTC)', y='Country', color='Sign-ins'),
          title=f'🕒 Sign-ins by hour x country - {geo_scope}').show()

print('Top countries by failed sign-ins:')
print(by_country.sort_values('Failed', ascending=False).head(10).to_string(index=False))

## 10. 🧪 Next-gen UEBA hardening
Closes the architectural gaps from the design review. Every module runs on the
**synthetic demo data** and adds columns to `agg` (no rework of v1). Each cell shows
the real Sentinel/KQL hook in comments so you can swap synthetic → production.

| § | Gap closed |
|---|---|
| 10b | Concept drift — PSI metric + retraining cadence |
| 10c | Cold-start fallback (thin-history accounts → peer/global baseline) |
| 10d | 2D UMAP/t-SNE embedding + SHAP beeswarm (analyst-readable) |
| 10e | Identity–asset node-link graph (blast radius) |
| 10f | Dynamic **resource-vector** peer clustering (function, not volume) |
| 10g | Workload / service-account (NHE) deterministic baselines |
| 10h | Active-learning feedback → IsoForest contamination/exclusion |
| 10i | Sequential ML — Markov event-transition modelling |

### 10b. Concept drift — Population Stability Index (PSI)
Unsupervised models decay as behaviour drifts (reorgs, holidays, new tooling). PSI
compares the score distribution of a **reference** run vs a **current** run.
**Rule of thumb:** `<0.1` stable · `0.1–0.25` investigate · `>0.25` retrain now.
**Cadence:** retrain weekly on a rolling 30-day lookback; alert when PSI > 0.25.

In [ ]:
# ── §10b: PSI drift detector — compares this run with the previous run stored in the Lakehouse ──
def psi(expected, actual, bins=10):
    qs = np.quantile(expected, np.linspace(0, 1, bins + 1)); qs[0] = -np.inf; qs[-1] = np.inf
    e = np.histogram(expected, qs)[0] / max(len(expected), 1)
    a = np.histogram(actual, qs)[0] / max(len(actual), 1)
    e = np.clip(e, 1e-4, None); a = np.clip(a, 1e-4, None)
    return float(np.sum((a - e) * np.log(a / e)))

cur = agg.iso_score.values
ref = None
if USE_SENTINEL:
    try:
        hist = spark.sql(f"SELECT RunId, iso_score FROM {OUTPUT_TABLE} WHERE RunId < '{RUN_ID}'").toPandas()
        if len(hist):
            prev = hist.RunId.max(); ref = hist[hist.RunId == prev].iso_score.values
            print('reference run:', prev)
    except Exception as e:
        print('no previous run available:', e)
if ref is None:
    _rng = np.random.default_rng(202); ref = cur; cur = cur + _rng.normal(0.05, 0.03, len(cur))
    print('first run / demo: simulating drift against this run')
val = psi(ref, cur)
status = 'OK (stable)' if val < 0.1 else ('WARN (investigate)' if val < 0.25 else 'CRITICAL → retrain')
print(f'PSI = {val:.3f}  →  {status}')
_d = pd.DataFrame({'score': np.r_[ref, cur], 'run': ['reference'] * len(ref) + ['current'] * len(cur)})
px.histogram(_d, x='score', color='run', barmode='overlay', nbins=40,
             title=f'Anomaly-score drift — PSI = {val:.3f} ({status})').show()

### 10c. Cold-start fallback — thin-history accounts
New users / fresh service accounts lack enough history for a stable personal baseline,
causing early false positives. Accounts with `< N_MIN_DAYS` of activity **inherit their
peer-group (or global) baseline** until their own matures. Adds `ActiveDays`,
`BaselineSource`, `ExpectedSignIns`.

In [ ]:
# ── §10c: Cold-start fallback ──
N_MIN_DAYS = 14
if not USE_SENTINEL or 'ActiveDays' not in agg or agg.ActiveDays.isna().all():
    agg['ActiveDays'] = np.random.default_rng(303).integers(1, 45, len(agg))   # demo only
agg['ActiveDays'] = agg['ActiveDays'].fillna(0).astype(int)                   # real: distinct active days from §1
_cold = agg.ActiveDays < N_MIN_DAYS
agg['BaselineSource'] = np.where(_cold, 'peer/global', 'individual')
_peer_exp = (agg.groupby('PeerGroup')['SignIns'].transform('median')
             if 'PeerGroup' in agg else pd.Series(agg['SignIns'].median(), index=agg.index))
agg['ExpectedSignIns'] = np.where(_cold, _peer_exp, agg['SignIns']).round(1)
print(agg.BaselineSource.value_counts().to_dict(), f'| threshold = {N_MIN_DAYS}d')
fig = px.scatter(agg, x='ActiveDays', y='Score', color='BaselineSource',
                 color_discrete_map={'peer/global': '#e67e22', 'individual': '#3498db'},
                 title=f'Cold-start handling — accounts < {N_MIN_DAYS}d use peer/global baseline')
fig.add_vline(x=N_MIN_DAYS, line_dash='dash', line_color='red'); fig.show()

### 10d. 2D embedding (UMAP/t-SNE) + SHAP beeswarm
3D PCA is hard to triage under pressure. A **2D UMAP** (t-SNE fallback) shows clusters
flat, and a **SHAP beeswarm** shows *which features* pushed each identity out of normal —
so a Tier-1 analyst sees the "why" instantly.

In [ ]:
# ── §10d: 2D embedding + SHAP beeswarm ──
_Xs = StandardScaler().fit_transform(X)
try:
    import umap
    emb = umap.UMAP(n_neighbors=min(15, len(agg) - 1), min_dist=0.1, random_state=42).fit_transform(_Xs); _m = 'UMAP'
except Exception:
    from sklearn.manifold import TSNE
    emb = TSNE(n_components=2, random_state=42, perplexity=min(30, len(agg) - 1)).fit_transform(_Xs); _m = 't-SNE (pip install umap-learn for UMAP)'
px.scatter(x=emb[:, 0], y=emb[:, 1], color=agg.Severity, color_discrete_map=cmap,
           hover_name=agg.UserPrincipalName,
           title=f'2D {_m} embedding — behavioural clusters (isolated points = anomalies)').show()

# SHAP beeswarm (falls back to a global bar if shap/matplotlib missing)
try:
    import shap
    _sv = shap.TreeExplainer(clf).shap_values(agg[z_cols].fillna(0))
    shap.summary_plot(_sv, agg[z_cols].fillna(0), plot_type='dot', show=True)
except Exception as _e:
    print('pip install shap matplotlib for the beeswarm — showing proxy importance bar')
    _imp = pd.Series(np.abs(_Xs).mean(0), index=z_cols).sort_values()
    px.bar(x=_imp.values, y=_imp.index, orientation='h', title='Feature importance (proxy)').show()

### 10e. Identity–asset node-link graph — blast radius
A timeline shows *when*; a graph shows *scope*. This maps the forensic `TARGET_USER`
(centre) to the **source IPs**, **countries**, and **apps/assets** they touched — failed
IPs in red — so an analyst sees lateral reach at a glance. Pure plotly, no networkx.

In [ ]:
# ── §10e: Identity–asset graph (blast radius) for TARGET_USER ──
import math
_ips = ev.IPAddress.value_counts().head(8)
_app = (ev.AppDisplayName.value_counts().head(6) if 'AppDisplayName' in ev.columns else pd.Series(dtype=int))
_ctry = ev.Country.value_counts().head(6)
_fail_ip = set(ev[ev.Failed].IPAddress.unique())

def _ring(items, radius, a0, a1, color, prefix):
    nodes = []
    n = max(len(items), 1)
    for i, (label, cnt) in enumerate(items.items()):
        ang = a0 + (a1 - a0) * (i / max(n - 1, 1))
        x, y = radius * math.cos(ang), radius * math.sin(ang)
        c = '#e74c3c' if (prefix == 'IP' and label in _fail_ip) else color
        nodes.append((f'{prefix}: {label}', x, y, cnt, c))
    return nodes

_nodes = ([('USER: ' + str(TARGET_USER), 0, 0, int(len(ev)), '#2c3e50')]
          + _ring(_ips, 1.0, math.pi * 0.6, math.pi * 1.4, '#9b59b6', 'IP')
          + _ring(_app, 1.0, -math.pi * 0.4, math.pi * 0.4, '#16a085', 'APP')
          + _ring(_ctry, 1.6, math.pi * 1.5, math.pi * 2.4, '#e67e22', 'GEO'))
_pos = {n[0]: (n[1], n[2]) for n in _nodes}

_ex, _ey = [], []
for label, x, y, *_ in _nodes[1:]:
    _ex += [0, x, None]; _ey += [0, y, None]
fig = go.Figure()
fig.add_trace(go.Scatter(x=_ex, y=_ey, mode='lines', line=dict(color='rgba(120,120,120,0.4)'), hoverinfo='none'))
fig.add_trace(go.Scatter(
    x=[n[1] for n in _nodes], y=[n[2] for n in _nodes], mode='markers+text',
    text=[n[0] for n in _nodes], textposition='top center',
    marker=dict(size=[max(14, min(46, 12 + n[3])) for n in _nodes], color=[n[4] for n in _nodes]),
    hovertext=[f'{n[0]} — {n[3]} events' for n in _nodes], hoverinfo='text'))
fig.update_layout(title=f'Identity–asset blast radius — {TARGET_USER} (red = failed-auth IPs)',
                  showlegend=False, height=620,
                  xaxis=dict(visible=False), yaxis=dict(visible=False)); fig.show()

### 10f. Dynamic resource-vector peer clustering
§8a clusters on **volume** features. People in the same department doing different jobs
still differ by **what they access**. Here we cluster on a per-identity **resource-access
vector** (frequency per app/resource) — function-based cohorts. Adds `ResourcePeerGroup`.

In [ ]:
# ── §10f: Resource-vector peer clustering (real app-access vectors when USE_SENTINEL) ──
from sklearn.cluster import KMeans
_apps = ['Office365', 'Teams', 'SharePoint', 'AzurePortal', 'Exchange', 'Salesforce', 'GitHub', 'SAP', 'LegacySMTP', 'PowerBI']
_arch = {'dev':     [.3, .2, .1, .6, .1, .0, .9, .1, .0, .2],
         'sales':   [.6, .5, .4, .1, .5, .9, .0, .1, .1, .6],
         'finance': [.5, .3, .5, .2, .4, .2, .0, .9, .2, .7],
         'admin':   [.4, .2, .3, .9, .3, .1, .4, .3, .6, .2]}
_rng = np.random.default_rng(606); _names = list(_arch)
_vecs = []
for _ in range(len(agg)):
    base = np.array(_arch[_rng.choice(_names, p=[.3, .3, .2, .2])])
    _vecs.append(np.clip(base + _rng.normal(0, 0.08, len(_apps)), 0, 1))
RV = pd.DataFrame(_vecs, columns=_apps, index=agg.UserPrincipalName)
if USE_SENTINEL:
    _acc = run_kql(f'''external_table('SigninLogs') | where TimeGenerated > ago({LOOKBACK}d) and ResultType == "0"
| summarize n=count() by UserPrincipalName, AppDisplayName''')
    _top = _acc.groupby('AppDisplayName').n.sum().nlargest(15).index
    _acc['App'] = np.where(_acc.AppDisplayName.isin(_top), _acc.AppDisplayName, 'Other')
    RV = (_acc.pivot_table(index='UserPrincipalName', columns='App', values='n', aggfunc='sum', fill_value=0)
              .reindex(agg.UserPrincipalName).fillna(0))
    RV = RV.div(RV.sum(axis=1).replace(0, 1), axis=0)          # share of each app per user
    _apps = list(RV.columns)
agg['ResourcePeerGroup'] = KMeans(n_clusters=min(4, max(2, len(agg) // 5)), n_init=10, random_state=42).fit_predict(StandardScaler().fit_transform(RV.values))
_prof = RV.groupby(agg.ResourcePeerGroup.values).mean()
px.imshow(_prof, x=_apps, aspect='auto', color_continuous_scale='Viridis',
          labels=dict(x='Resource/App', y='Resource peer group', color='Mean usage'),
          title='Resource-access peer groups (function-based, not volume)').show()
print('Volume (§8a) vs resource (§10f) peer-group cross-tab:')
print(pd.crosstab(agg.get('PeerGroup', pd.Series(0, index=agg.index)), agg.ResourcePeerGroup))

### 10g. Workload / service-account (NHE) baselines
Most identity attacks now target **non-human entities** (service accounts, managed
identities, OAuth apps). NHEs should be **highly predictable**, so they get a
**deterministic low-tolerance** baseline (tighter z threshold) than variable humans.
Adds `IdentityType`, `TypeThreshold`, `TypeAnomaly`.

In [ ]:
# ── §10g: Human vs Non-Human Entity (NHE) baselines ──
_NHE_TOKENS = ('svc', 'svc-', 'sa-', 'app-', 'service', 'daemon', 'automation', '-mi', 'managedidentity', 'oauth')
agg['IdentityType'] = np.where(
    agg.UserPrincipalName.str.lower().apply(lambda u: any(t in u for t in _NHE_TOKENS)), 'NHE', 'Human')
# DEMO: promote a few accounts to NHE so the branch is exercised
if not USE_SENTINEL:
    _demo_nhe = agg.sample(min(15, len(agg)), random_state=7).index
    agg.loc[_demo_nhe, 'IdentityType'] = 'NHE'

HUMAN_Z, NHE_Z = 3.0, 1.5                                # NHEs: low tolerance (should be deterministic)
agg['TypeThreshold'] = np.where(agg.IdentityType == 'NHE', NHE_Z, HUMAN_Z)
_dev = agg['PeerZ'] if 'PeerZ' in agg else agg.get('SignIns_z', pd.Series(0, index=agg.index))
agg['TypeAnomaly'] = (_dev >= agg.TypeThreshold).astype(int)
print(agg.IdentityType.value_counts().to_dict(),
      '| NHE anomalies:', int(agg[agg.IdentityType == 'NHE'].TypeAnomaly.sum()),
      '| Human anomalies:', int(agg[agg.IdentityType == 'Human'].TypeAnomaly.sum()))
px.scatter(agg, x='IdentityType', y=_dev, color='TypeAnomaly', color_continuous_scale='Reds',
           hover_name='UserPrincipalName',
           title='Deviation vs identity type — NHEs judged on a tighter (1.5σ) threshold').show()

### 10h. Active-learning feedback → the model layer
§8g feedback only dampened the *score*; the **model** kept flagging the same FPs. Here
analyst dispositions drive the unsupervised layer: FP/benign identities are **excluded
from the fit** and the IsoForest **contamination** is adapted, so repeat false positives
are structurally suppressed. Adds `iso_anomaly_AL`.

In [ ]:
# ── §10h: Semi-supervised active learning into IsoForest ──
_FB = dict(FEEDBACK) if ('FEEDBACK' in dir() and FEEDBACK) or USE_SENTINEL else {
    'user099@contoso.com': 'fp', 'user250@contoso.com': 'benign', 'user007@contoso.com': 'tp'}  # DEMO
_fp_users = [u for u, v in _FB.items() if v in ('fp', 'benign')]
_base_contam = 0.02
_adj_contam = float(np.clip(_base_contam * (1 - 0.1 * len(_fp_users)), 0.005, 0.05))
_fit_mask = ~agg.UserPrincipalName.isin(_fp_users)       # exclude confirmed FP/benign from training
clf_al = IsolationForest(n_estimators=200, contamination=_adj_contam, random_state=42, n_jobs=-1).fit(X[_fit_mask.values])
agg['iso_anomaly_AL'] = clf_al.predict(X)
_before = int((agg.iso_anomaly == -1).sum()); _after = int((agg.iso_anomaly_AL == -1).sum())
print(f'contamination {_base_contam} → {_adj_contam:.3f} | flags {_before} → {_after} | excluded FP/benign: {_fp_users}')
_cmp = pd.DataFrame({'stage': ['v1 IsoForest', 'after active-learning'], 'flags': [_before, _after]})
px.bar(_cmp, x='stage', y='flags', color='stage', title='False-positive suppression via analyst feedback').show()

### 10i. Sequential ML — Markov event-transition modelling
Tabular models treat events as independent. Attacks are **sequences** (A→B→C). We learn
a first-order **Markov transition matrix** from normal action chains, then score each
identity's sequence by **negative log-likelihood** — flagging anomalous *order* even when
each single action looks benign. Adds `SeqNLL`, `SeqAnomaly`. (Autoencoders are the next step.)

In [ ]:
# ── §10i: Markov sequence anomaly model ──
# Real data: each user's ordered sign-in sequence of states "<app>|<ok/fail>" (top apps, others = Other).
# Learn a fleet-wide transition matrix, then score each user's sequence by negative log-likelihood.
if USE_SENTINEL:
    _s = run_kql(f'''external_table('SigninLogs') | where TimeGenerated > ago({LOOKBACK}d)
| project TimeGenerated, UserPrincipalName, AppDisplayName, Ok = iff(tostring(ResultType) == "0", "ok", "fail")''')
    _s = _s.sort_values(['UserPrincipalName', 'TimeGenerated'])
    _topapps = _s.AppDisplayName.value_counts().head(8).index
    _s['State'] = np.where(_s.AppDisplayName.isin(_topapps), _s.AppDisplayName, 'Other') + '|' + _s.Ok
    _seqs = {u: g.State.tolist() for u, g in _s.groupby('UserPrincipalName') if len(g) >= 3}
else:
    _acts0 = ['login', 'mfa', 'read', 'list', 'download', 'admin', 'export', 'logout']
    _i0 = {a: i for i, a in enumerate(_acts0)}; _rng = np.random.default_rng(909)
    _T0 = np.full((8, 8), 0.02)
    for a, b in [('login','mfa'),('mfa','read'),('read','list'),('list','read'),('read','download'),('download','logout'),('list','logout'),('read','logout')]:
        _T0[_i0[a], _i0[b]] = 0.5
    _T0 = _T0 / _T0.sum(1, keepdims=True)
    def _gen(anom=False, n=30):
        s, cur = ['login'], 'login'
        for _ in range(n - 1):
            nxt = _rng.choice(['admin', 'export', 'download']) if (anom and _rng.random() < 0.25) else _rng.choice(_acts0, p=_T0[_i0[cur]])
            s.append(nxt); cur = nxt
        return s
    _known = {'user007@contoso.com', 'user099@contoso.com', 'user250@contoso.com'}
    _seqs = {u: _gen(anom=(u in _known)) for u in agg.UserPrincipalName}

_acts = sorted({x for s in _seqs.values() for x in s})
_idx = {a: i for i, a in enumerate(_acts)}
_M = np.ones((len(_acts), len(_acts)))                    # Laplace-smoothed fleet transition model
for seq in _seqs.values():
    for a, b in zip(seq, seq[1:]):
        _M[_idx[a], _idx[b]] += 1
_M = _M / _M.sum(1, keepdims=True)

def _nll(seq):
    return float(-np.mean([np.log(_M[_idx[a], _idx[b]]) for a, b in zip(seq, seq[1:])])) if len(seq) > 1 else 0.0

_seq = pd.DataFrame([{'UserPrincipalName': u, 'SeqNLL': round(_nll(s), 3)} for u, s in _seqs.items()])
_thr = _seq.SeqNLL.mean() + 2 * _seq.SeqNLL.std()
_seq['SeqAnomaly'] = (_seq.SeqNLL > _thr).astype(int)
agg = agg.drop(columns=[c for c in ['SeqNLL', 'SeqAnomaly'] if c in agg]).merge(_seq, on='UserPrincipalName', how='left')
print(f'Sequence anomalies (NLL > mean+2σ): {int(_seq.SeqAnomaly.sum())} | threshold = {_thr:.2f} | states = {len(_acts)}')
px.imshow(_M, x=_acts, y=_acts, color_continuous_scale='Blues',
          labels=dict(x='next state', y='current state', color='P'),
          title='Learned sign-in transition matrix (fleet behaviour)').show()
px.histogram(_seq, x='SeqNLL', nbins=40, title='Sequence anomaly score (neg-log-likelihood) — higher = stranger order').show()
_seq.sort_values('SeqNLL', ascending=False).head(10)

## 11. Save the full enriched results (Fabric)
Writes every computed column (priority, confidence, narrative, peer groups, NHE, sequence score…) to `IdentityAnomalyScores_Detail`, overwritten each run, for dashboards and Sentinel federation.

In [ ]:
if USE_SENTINEL:
    _cols = [c for c in ['UserPrincipalName','Score','Severity','PriorityScore','Confidence','CalibratedRisk','PeerGroup','PeerZ','TwoTailedZ',
                         'SuppressionFlag','AssetCriticality','ExposureScore','AdjExposure','AnalystVerdict','ActiveDays','BaselineSource',
                         'ResourcePeerGroup','IdentityType','TypeAnomaly','iso_anomaly_AL','SeqNLL','SeqAnomaly','Narrative'] if c in agg.columns]
    det = agg[_cols].copy()
    for c in det.columns:
        if det[c].dtype == 'object' or str(det[c].dtype) == 'category': det[c] = det[c].astype(str)
    det['TimeGenerated'] = pd.Timestamp.utcnow().tz_localize(None); det['RunId'] = RUN_ID
    spark.createDataFrame(det).write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(OUTPUT_TABLE + '_Detail')
    print(f'wrote {len(det)} rows -> {OUTPUT_TABLE}_Detail')
else:
    print('demo mode: nothing written')